## Setup — Librerías permitidas

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from functools import reduce
from scipy import stats

try:
    import missingno as msno
except ImportError:
    msno = None

## Punto 3 — Comparación de 3 datasets candidatos

> Ejecutá primero la celda de **imports** de arriba.

In [10]:
datos_comparacion = {
    "Criterio": [
        "Nombre",
        "Volumen (Filas)",
        "Estructura de Datos",
        "Variables Temporales / Geo",
        "Ventaja Principal",
        "Motivo de la Decisión"
    ],
    "Candidato 1 (Seleccionado)": [
        "Brazilian E-Commerce (Olist)",
        "> 99,000 pedidos",
        "Relacional (9 tablas interconectadas)",
        "Fechas exactas (horas) y regiones de Brasil",
        "Alta complejidad técnica; permite cruzar logística, pagos y geografía",
        "ELEGIDO. Su estructura imita entornos reales y permite análisis multivariado profundo."
    ],
    "Candidato 2 (Descartado)": [
        "Customer Shopping Dataset",
        "~ 99,000 registros",
        "Tabla plana única",
        "Solo fechas básicas, sin horas",
        "Datos limpios y listos para usar sin necesidad de hacer merges",
        "DESCARTADO. Demasiado simple; no presenta desafíos en la preparación de datos."
    ],
    "Candidato 3 (Descartado)": [
        "Retail Sales Data",
        "Variable",
        "Tabla plana única",
        "Tendencias estacionales genéricas",
        "Bueno para análisis de series de tiempo básico",
        "DESCARTADO. Variables limitadas que restringen las preguntas analíticas complejas."
    ]
}

df_comparacion = pd.DataFrame(datos_comparacion)
df_comparacion

,Criterio,Candidato 1 (Seleccionado),Candidato 2 (Descartado),Candidato 3 (Descartado)
0,Nombre,Brazilian E-Commerce (Olist),Customer Shopping Dataset,Retail Sales Data
1,Volumen (Filas),"> 99,000 pedidos","~ 99,000 registros",Variable
2,Estructura de Datos,Relacional (9 tablas interconectadas),Tabla plana única,Tabla plana única
3,Variables Temporales / Geo,Fechas exactas (horas) y regiones de Brasil,"Solo fechas básicas, sin horas",Tendencias estacionales genéricas
4,Ventaja Principal,Alta complejidad técnica; permite cruzar logís...,Datos limpios y listos para usar sin necesidad...,Bueno para análisis de series de tiempo básico
5,Motivo de la Decisión,ELEGIDO. Su estructura imita entornos reales y...,DESCARTADO. Demasiado simple; no presenta desa...,DESCARTADO. Variables limitadas que restringen...


## Pregunta Analítica [0.3 puntos]

Esta pregunta sigue la estructura exigida para el Solemne, enfocándose en el cruce de **logística**, **geografía** y **satisfacción** del cliente.

**Pregunta Analítica:**

> ¿Qué patrones, anomalías y relaciones relevantes se observan entre los **tiempos reales de entrega logísticos**, las **regiones geográficas de Brasil** y la **satisfacción final del usuario** (review scores), y qué **decisiones preliminares** sobre la optimización de rutas de envío y políticas de retención de clientes se pueden recomendar a partir de este análisis exploratorio?

## Criterios de Éxito del Análisis [0.3 puntos]

Los criterios de éxito definen cómo evaluaremos si la pregunta analítica fue respondida. Se organizan en tres dimensiones: **Técnica**, **Analítica** y **Negocio**.

### A. Dimensión Técnica (Preparación de Datos)

1. Consolidar con éxito un **DataFrame Maestro** uniendo al menos 4 tablas relacionales de Olist (`orders`, `order_items`, `order_reviews` y `customers`) sin generar registros duplicados erróneos por culpa de las relaciones de uno a muchos.
2. Identificar, documentar y tratar los **valores nulos o atípicos (outliers)** en las columnas críticas, específicamente en las fechas de entrega y los costos de flete (`freight_value`).

### B. Dimensión Analítica (Exploración)

1. Crear al menos una **variable nueva calculada** (Feature Engineering), como por ejemplo: `dias_retraso` (diferencia entre fecha de entrega estimada y real).
2. Demostrar mediante **gráficos bivariados** (ej. Boxplots o Scatterplots) la relación entre el tiempo de espera del cliente y la calificación de su reseña (de 1 a 5 estrellas).
3. Identificar visualmente y agrupar qué **estados (regiones) de Brasil** concentran los mayores volúmenes de ventas y los peores tiempos de entrega.

### C. Dimensión de Negocio (Accionabilidad)

1. Generar al menos **dos recomendaciones concretas** respaldadas por los datos empíricos (ej. *"Reasignar proveedores logísticos en el estado X debido a una caída del 40% en la satisfacción por retrasos"*).

In [ ]:
pregunta_analitica = (
    "¿Qué patrones, anomalías y relaciones relevantes se observan entre los tiempos "
    "reales de entrega logísticos, las regiones geográficas de Brasil y la satisfacción "
    "final del usuario (review scores), y qué decisiones preliminares sobre la "
    "optimización de rutas de envío y políticas de retención de clientes se pueden "
    "recomendar a partir de este análisis exploratorio?"
)

criterios_exito = {
    "Técnica": [
        "Consolidar un DataFrame Maestro uniendo al menos 4 tablas (orders, order_items, order_reviews, customers) sin duplicados erróneos.",
        "Identificar, documentar y tratar nulos/outliers en fechas de entrega y freight_value.",
    ],
    "Analítica": [
        "Crear al menos una variable calculada (ej. dias_retraso).",
        "Gráficos bivariados que relacionen tiempo de espera y review score.",
        "Identificar estados con mayor volumen de ventas y peores tiempos de entrega.",
    ],
    "Negocio": [
        "Al menos dos recomendaciones concretas respaldadas por los datos empíricos.",
    ],
}

print("Pregunta analítica:")
print(pregunta_analitica)

print("\nCriterios de éxito:")
for dimension, criterios in criterios_exito.items():
    print(f"\n{dimension}:")
    for i, criterio in enumerate(criterios, start=1):
        print(f"  {i}. {criterio}")

## Punto 4 — Diccionario de datos del dataset maestro

In [11]:
BASE = "https://raw.githubusercontent.com/spdrio/Brazilian-E-Commerce-Public-Dataset-by-Olist/master/files"

def obtener_dataset_maestro():
    try:
        orders = pd.read_csv(f"{BASE}/olist_orders_dataset.csv")
        customers = pd.read_csv(f"{BASE}/olist_customers_dataset.csv")
        payments = pd.read_csv(f"{BASE}/olist_order_payments_dataset.csv")

        df_master = orders.merge(customers, on="customer_id", how="inner")
        df_master = df_master.merge(payments, on="order_id", how="left")
        return df_master

    except FileNotFoundError:
        print("(Aviso: CSV no encontrados. Usando esquema teórico...)\n")
        return pd.DataFrame({
            "order_id": pd.Series(dtype="object"),
            "customer_id": pd.Series(dtype="object"),
            "order_status": pd.Series(dtype="object"),
            "order_purchase_timestamp": pd.Series(dtype="datetime64[ns]"),
            "customer_state": pd.Series(dtype="object"),
            "payment_type": pd.Series(dtype="object"),
            "payment_value": pd.Series(dtype="float64"),
        })

def generar_diccionario(df):
    diccionario = pd.DataFrame({
        "Nombre de la Columna": df.columns,
        "Tipo de Dato (Python)": df.dtypes.astype(str),
    })

    descripciones = {
        "order_id": "Código alfanumérico único para cada pedido.",
        "customer_id": "Código alfanumérico único del cliente en ese pedido.",
        "order_status": "Estado actual del pedido (entregado, cancelado, facturado, etc.).",
        "order_purchase_timestamp": "Fecha y hora exacta en que se realizó la transacción.",
        "customer_state": "Sigla del estado (región) de residencia del cliente (ej. SP, RJ).",
        "payment_type": "Método de pago utilizado (credit_card, boleto, voucher).",
        "payment_value": "Monto total pagado en esa transacción.",
    }

    tipo_rubrica = {
        "order_id": "Identificador",
        "customer_id": "Identificador",
        "order_status": "Categórica Nominal",
        "order_purchase_timestamp": "Temporal",
        "customer_state": "Geográfica",
        "payment_type": "Categórica Nominal",
        "payment_value": "Numérica Continua",
    }

    diccionario["Tipo de Variable (Rúbrica)"] = (
        diccionario["Nombre de la Columna"].map(tipo_rubrica).fillna("Desconocido")
    )
    diccionario["Descripción"] = (
        diccionario["Nombre de la Columna"].map(descripciones).fillna("Sin descripción")
    )

    return diccionario[
        ["Nombre de la Columna", "Tipo de Dato (Python)", "Tipo de Variable (Rúbrica)", "Descripción"]
    ]

df_maestro = obtener_dataset_maestro()
df_diccionario = generar_diccionario(df_maestro)
df_diccionario

,Nombre de la Columna,Tipo de Dato (Python),Tipo de Variable (Rúbrica),Descripción
order_id,order_id,str,Identificador,Código alfanumérico único para cada pedido.
customer_id,customer_id,str,Identificador,Código alfanumérico único del cliente en ese p...
order_status,order_status,str,Categórica Nominal,"Estado actual del pedido (entregado, cancelado..."
order_purchase_timestamp,order_purchase_timestamp,str,Temporal,Fecha y hora exacta en que se realizó la trans...
order_approved_at,order_approved_at,str,Desconocido,Sin descripción
order_delivered_carrier_date,order_delivered_carrier_date,str,Desconocido,Sin descripción
order_delivered_customer_date,order_delivered_customer_date,str,Desconocido,Sin descripción
order_estimated_delivery_date,order_estimated_delivery_date,str,Desconocido,Sin descripción
customer_unique_id,customer_unique_id,str,Desconocido,Sin descripción
customer_zip_code_prefix,customer_zip_code_prefix,int64,Desconocido,Sin descripción
